# Chat 26 — AI concepts lab

Five small experiments on **real JobPulse data**: the live pipeline's output for `snapshot_date=2026-10-09`
(silver jobs + the Voyage vectors the embedding job wrote that night). Nothing here writes to AWS.

| Step | Concept | Question it answers |
|---|---|---|
| 1 | Embeddings + cosine similarity | What *is* a vector, and what does "similar" mean in numbers? |
| 2 | Keyword (BM25) vs vector search | When does each one win? Why do people build *hybrid* search? |
| 3 | Chunking + truncation | What does the 4,000-char cut in `embedding_agent.py` actually throw away? |
| 4 | RAG (retrieval-augmented generation) | Does giving Claude the job text change the answer? |
| 5 | Precision@10 | How do you put *one number* on "search is good"? |

**Run it:** `.venv` in the repo root (see `docs/runbook.md` → local lab). Cost of one full run: Voyage ≈ 30 short texts
(free tier), Claude Haiku 5.5 ≈ 3 calls (< $0.01).

In [1]:
import html
import json
import re
from pathlib import Path

import anthropic
import boto3
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import voyageai
import yaml
from IPython.display import Markdown, display
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 80)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = Path(r"C:\Users\malik\jobpulse_lab_data")  # local copy, outside the repo
DATE = "2026-10-09"
MODEL = "voyage-4-lite"  # same as genai/embedding_agent.py
MAX_CHARS = 4000         # same cut as genai/embedding_agent.py
REGION = "ap-south-1"

# --- download once (read-only S3 GETs, ~9 MB) ---
s3 = boto3.client("s3", region_name=REGION)
if not (DATA / f"embeddings_{DATE}.parquet").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    s3.download_file("jobpulse-gold-dev", f"embeddings/snapshot_date={DATE}/data.parquet", str(DATA / f"embeddings_{DATE}.parquet"))
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket="jobpulse-silver-dev", Prefix=f"snapshot_date={DATE}/"):
        for obj in page.get("Contents", []):
            local = DATA / f"silver_{DATE}" / obj["Key"].split("/", 1)[1]
            local.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file("jobpulse-silver-dev", obj["Key"], str(local))

silver = ds.dataset(DATA / f"silver_{DATE}", format="parquet", partitioning="hive").to_table().to_pandas()
silver["job_id"] = silver["job_id"].astype(str)
emb = pd.read_parquet(DATA / f"embeddings_{DATE}.parquet")
jobs = emb.merge(silver.drop_duplicates("job_id"), on="job_id")  # only jobs that have a vector
M = np.array(jobs["embedding"].tolist(), dtype=np.float32)       # one row per job

# --- API clients: keys come from Secrets Manager, never printed ---
sm = boto3.client("secretsmanager", region_name=REGION)
def secret(name, field):
    return json.loads(sm.get_secret_value(SecretId=name)["SecretString"])[field]

vo = voyageai.Client(api_key=secret("jobpulse/voyage_key_dev", "VOYAGE_API_KEY"))
claude = anthropic.Anthropic(api_key=secret("jobpulse/anthropic_key_dev", "ANTHROPIC_API_KEY"))

def embed(texts):
    return np.array(vo.embed(texts, model=MODEL).embeddings, dtype=np.float32)

def strip_html(text):
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", text))).strip()

print(f"silver rows: {len(silver):,} | rows with a vector: {len(jobs):,} | vector matrix: {M.shape}")
print("rows with a vector, by source:", jobs.groupby("source").size().to_dict())
print("silver rows with an EMPTY description, by source:",
      silver[silver.description.fillna("") == ""].groupby("source").size().to_dict())

C:\Users\malik\my_proj\jobpulse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


silver rows: 6,006 | rows with a vector: 1,544 | vector matrix: (1544, 1024)
rows with a vector, by source: {'adzuna': 1200, 'arbeitnow': 325, 'remotive': 19}
silver rows with an EMPTY description, by source: {'greenhouse': 4462}


**Read this before going on.** Only 1,544 of 6,006 jobs have a vector. Greenhouse sends no description at all
(that's the Chat 27 prerequisite: `?content=true`), and 1,200 of the 1,544 are **Adzuna 500-character snippets**.
So today's "semantic search" mostly searches snippets.

---
## Step 1 — What an embedding is

An **embedding** is a list of numbers that stands for the *meaning* of a text. Texts with similar meaning get similar
lists. Think of it like a Spark `VectorAssembler` output column — except a neural network picks the features, and we
can't name them.

**Cosine similarity** is the angle between two vectors: 1.0 = same direction (same meaning), 0 = unrelated.
Voyage returns vectors of length 1, so cosine is just the dot product `a @ b`.

We take 5 jobs: two data engineers (one in French), a backend engineer, and two Adzuna snippets from other fields.

In [2]:
picks = {
    "Senior Data Engineer · Aqemia (EN)":  "senior-data-engineer-paris-96158",
    "Data Engineer GCP · SFEIR (FR)":      "data-engineer-gcp-paris-150905",
    "Backend Engineer · Kpler (EN)":       "backend-engineer-commodities-united-kingdom-202132",
    "Financial Accountant (snippet)":      "5902274680",
    "Receptionist (snippet)":              "5909215709",
}
five = jobs.set_index("job_id").loc[list(picks.values())]

# Re-embed exactly like the pipeline does: raw description, first 4,000 chars, same model.
fresh = embed([d[:MAX_CHARS] for d in five["description"]])
stored = np.array(five["embedding"].tolist(), dtype=np.float32)

print("numbers per vector:", fresh.shape[1], "(the code comments say 512 — the data says 1024)")
print("first 8 numbers of the Aqemia vector:", np.round(fresh[0][:8], 4))
print("length of each vector:", np.round(np.linalg.norm(fresh, axis=1), 4))
print("fresh vs stored-by-pipeline cosine:", np.round((fresh * stored).sum(axis=1), 4))

short = [k.split(" ·")[0].split(" (")[0] for k in picks]
pd.DataFrame(fresh @ fresh.T, index=short, columns=short).round(2)

numbers per vector: 1024 (the code comments say 512 — the data says 1024)
first 8 numbers of the Aqemia vector: [ 0.0003  0.0244  0.0547  0.0543 -0.0119  0.0144 -0.0213  0.0421]
length of each vector: [1. 1. 1. 1. 1.]
fresh vs stored-by-pipeline cosine: [1. 1. 1. 1. 1.]


,Senior Data Engineer,Data Engineer GCP,Backend Engineer,Financial Accountant,Receptionist
Senior Data Engineer,1.00,0.55,0.56,0.51,0.33
Data Engineer GCP,0.55,1.00,0.48,0.36,0.26
Backend Engineer,0.56,0.48,1.00,0.41,0.29
Financial Accountant,0.51,0.36,0.41,1.00,0.51
Receptionist,0.33,0.26,0.29,0.51,1.00


**What to notice**
- 1,024 numbers, each meaningless alone. Meaning lives in the *pattern* — like one row of a feature table.
- Fresh vs stored ≈ 1.0 → the pipeline's vectors are reproducible: same model, same text in. If someone changed the
  model, this would drop — **never mix vectors from two models in one index**.
- **Surprise:** the Aqemia *Senior Data Engineer* is about as close to the Backend Engineer (0.56) and even the
  Accountant (0.51) as to the other Data Engineer (0.55). The vector isn't wrong — it faithfully encodes the text it
  was given, and Step 3 shows that text is mostly a drug-company blurb. **Garbage in, garbage vector.**
- The two admin snippets (Accountant, Receptionist) pair up at 0.51; the Receptionist is far from every tech job (~0.3).
- The scores are squeezed into a narrow band (unrelated jobs still score ~0.3–0.5). Only the *ranking* means anything —
  never use a fixed threshold like "> 0.7 = match" without checking on real data.

---
## Step 2 — Keyword search vs vector search

**BM25** = classic keyword ranking (what Elasticsearch / Postgres full-text do): it counts matching words, rewards rare
words, and normalises for document length. It knows *nothing* about meaning.
**Vector search** = embed the query, rank jobs by cosine. It knows meaning but is fuzzy on exact names.

Both indexes below see the same text (first 4,000 chars of the description) over all 1,544 jobs.

In [3]:
def tokens(text):
    return re.findall(r"[a-z0-9+#.]+", text.lower())

bm25 = BM25Okapi([tokens(d[:MAX_CHARS]) for d in jobs["description"]])

def compare(query, k=5):
    q = embed([query])[0]
    vec_top = np.argsort(-(M @ q))[:k]
    kw_scores = bm25.get_scores(tokens(query))
    kw_top = np.argsort(-kw_scores)[:k]
    display(Markdown(f"**Query:** `{query}`"))
    display(pd.DataFrame({
        "vector (meaning)": jobs["title"].iloc[vec_top].values,
        "BM25 (exact words)": [jobs["title"].iloc[i] if kw_scores[i] > 0 else "— no job contains these words —" for i in kw_top],
    }, index=range(1, k + 1)))

display(Markdown("### Vector wins — the words differ, the meaning matches"))
for q in ["someone who keeps our servers and deployments running",
          "work with numbers and spreadsheets for the finance team",
          "talking to customers on the phone and fixing their issues"]:
    compare(q)

display(Markdown("### Keyword wins — exact tool names"))
for q in ["dbt", "Snowflake", "Airflow"]:
    compare(q)

### Vector wins — the words differ, the meaning matches

**Query:** `someone who keeps our servers and deployments running`

,vector (meaning),BM25 (exact words)
1,Infra Tech Support Practitioner,"Controls Engineer, EMEA Controls"
2,ISSRRN000152 - Application Tech Support Practitioner Copy 02,Operations associate
3,Cloud Engineer,Embedded C Developer (CPT In-office)
4,Applications Support-Production Support-L2,Senior Automation & OT Engineer e/v/e/r/y/o/n/e
5,Director of Engineering - Infrastructure Operations,Duty Manager: Front Of House &amp; Housekeeping - Thornybush


**Query:** `work with numbers and spreadsheets for the finance team`

,vector (meaning),BM25 (exact words)
1,Regional Financial Operations Analyst,Account Operations & Farming Specialist
2,Financial Accountant,Financial Controller
3,Front Office & Finance Assistant,Senior Data & Business Analyst - Finance (DAR)
4,Senior Accountant,Financial Accountant
5,Accountant,Finance Associate


...

,vector (meaning),BM25 (exact words)
1,Relationship Manager - Retail,DATA ENGINEER*IN (m/w/d)
2,PostgreSQL Database Administrator (DBA) - 5 years - Pune - Full Time,ANALYTICS ENGINEER*IN (m/w/d)
3,Intern- Trainee Developer,Lead Data Engineer GCP
4,Legal Secretary,Data Engineer GCP
5,ANALYTICS ENGINEER*IN (m/w/d),Cloud & Data Engineer


**Query:** `Snowflake`

,vector (meaning),BM25 (exact words)
1,Relationship Manager - Retail,AI / ML Engineer
2,Peter Alexander - Christmas Angels - Canelands,Data Engineer expérimenté(e)
3,AI / ML Engineer,Data Engineer Confirmé(e)
4,Witchery - Seasonal Sales Consultant - WA,Lead Data Engineer GCP
5,"CONNOR Christmas Casual - Burwood, NSW",Data Engineer GCP


**Query:** `Airflow`

,vector (meaning),BM25 (exact words)
1,Sr Mech Design Eng Manager,DATA ENGINEER*IN (m/w/d)
2,Mech Design Engr II,Data Engineer (F/H/NB)
3,Assistant Manager Sales,Data Engineer (F/H/NB)
4,Gas Operative,Backend Engineer - Commodities
5,Financial Analyst,Data Engineer expérimenté(e)


**What to notice**
- "keeps our servers running" → vector finds infra/support roles; BM25 matches the word *keeps* and finds **housekeeping**.
- "Snowflake" → vector finds **Christmas casual staff** (snow 🎄), "Airflow" → **mechanical engineers** (air flow).
  A one-word query has almost no context, so the vector falls back to the everyday meaning. BM25 just finds the word.
- "dbt" → the vector has no idea what *dbt* means in this context; BM25 lists data engineers.

This is why production search is **hybrid**: run both, then merge the two ranked lists. JobPulse's dashboard does a
0.5/0.5 score blend today; Chat 28 tests **RRF** (Reciprocal Rank Fusion — merge by *rank position*, not raw score,
because BM25 scores like 20.1 and cosines like 0.6 aren't on the same scale).

---
## Step 3 — Chunking, and what the 4,000-char cut throws away

`embedding_agent.py` sends `description[:4000]` — one vector per job. Two problems hide in that one line:
1. Arbeitnow/Remotive descriptions are **raw HTML**, so part of those 4,000 chars is `<span style="color: #000000;">`.
2. Long JDs put the company blurb first. The skills come later — after the cut.

**Chunking** = split a document into pieces (sections, paragraphs) and embed each piece separately, so a query can
match the *part* that's about it.

In [4]:
jd = jobs.set_index("job_id").loc["senior-data-engineer-paris-96158"]
raw = jd["description"]
headings = ["About AQEMIA", "About our Team", "About our Engineering Department", "The role", "Responsibilities",
            "Qualifications", "Nice-to-have", "Our recruitment process", "Why Join Us?"]

starts = [raw.find(">" + h) + 1 for h in headings]  # heading text right after an HTML tag
assert starts == sorted(starts) and min(starts) > 0, starts

sec = pd.DataFrame({"section": headings, "raw_start": starts})
sec["raw_end"] = sec["raw_start"].shift(-1, fill_value=len(raw))
sec["text"] = [strip_html(raw[a:b]) for a, b in zip(sec.raw_start, sec.raw_end)]
sec["visible_chars"] = sec.text.str.len()
sec["in_pipeline_vector"] = np.select([sec.raw_end <= MAX_CHARS, sec.raw_start < MAX_CHARS], ["yes", "partly"], "NO — cut")
print(f"raw length {len(raw):,} chars, visible text {len(strip_html(raw)):,} chars; pipeline embeds raw[:{MAX_CHARS}]")
sec[["section", "raw_start", "visible_chars", "in_pipeline_vector"]]

raw length 9,143 chars, visible text 7,706 chars; pipeline embeds raw[:4000]


,section,raw_start,visible_chars,in_pipeline_vector
0,About AQEMIA,12,1148,yes
1,About our Team,1587,676,yes
2,About our Engineering Department,2348,860,yes
3,The role,3332,986,partly
4,Responsibilities,4379,1289,NO — cut
5,Qualifications,5770,646,NO — cut
6,Nice-to-have,6515,536,NO — cut
7,Our recruitment process,7144,399,NO — cut
8,Why Join Us?,7645,1158,NO — cut


In [5]:
S = embed(sec["text"].tolist())  # one vector per section (= chunk)
queries = ["must have Python, SQL and Airflow experience",
           "what will I do day to day in this job",
           "can I work from London or remotely"]
Q = embed(queries)

table = pd.DataFrame(S @ Q.T, index=sec["section"], columns=queries)
table.loc["▶ whole JD, pipeline vector (first 4,000 raw chars)"] = Q @ stored[0]
table = table.round(3).astype(object)
table.loc["▶ best section"] = table.iloc[:-1].astype(float).idxmax().values
table

,"must have Python, SQL and Airflow experience",what will I do day to day in this job,can I work from London or remotely
section,,,
About AQEMIA,0.355,0.069,0.133
About our Team,0.425,0.133,0.337
About our Engineering Department,0.459,0.14,0.213
The role,0.521,0.222,0.203
Responsibilities,0.495,0.232,0.174
Qualifications,0.709,0.16,0.255
Nice-to-have,0.584,0.16,0.25
Our recruitment process,0.366,0.209,0.253
Why Join Us?,0.423,0.173,0.355


In [6]:
print("LAST 300 visible chars the pipeline vector still sees:\n ", strip_html(raw[:MAX_CHARS])[-300:])
print("\nFIRST 500 visible chars it throws away:\n ", strip_html(raw[MAX_CHARS:])[:500])

long = silver[silver.description.fillna("").str.len() > 0].copy()
long["raw_chars"] = long.description.str.len()
long["visible_in_first_4000"] = long.description.str[:MAX_CHARS].map(strip_html).str.len()
long["cut_off"] = long.raw_chars > MAX_CHARS
long.groupby("source").agg(jobs=("raw_chars", "size"), median_raw_chars=("raw_chars", "median"),
                           median_visible_in_first_4000=("visible_in_first_4000", "median"),
                           share_cut_off=("cut_off", "mean")).round(2)

LAST 300 visible chars the pipeline vector still sees:
  perimental results from CROs and partners. A core part of the work is modelling these scientific entities well — establishing canonical identity, provenance and trustworthy lineage across heterogeneous, often messy sources — so scientists, and increasingly AI agents, can rely on them. You'll work at

FIRST 500 visible chars it throws away:
  the intersection of software engineering, data infrastructure and scientific research, with real scope to shape architecture rather than just execute against it. As AQEMIA moves toward more service and API-driven integration next year, you'll help make data fit for automation — expanding your impact from pipelines to the systems that consume them. Responsibilities Own AQEMIA's Bronze → Silver → Gold data pipelines end to end, from ingestion through transformation and delivery, maintaining lineag


,jobs,median_raw_chars,median_visible_in_first_4000,share_cut_off
source,,,,
adzuna,1200,500.0,500.0,0.00
arbeitnow,325,6346.0,2988.0,0.79
remotive,19,4980.0,2452.0,0.63


**What to notice**
- The Aqemia vector is built from *About AQEMIA / Team / Engineering / The role*. **Responsibilities and
  Qualifications — the part a job seeker searches for — are not in it.**
- Section vectors put each query on the right section (skills → Qualifications, day-to-day → Responsibilities,
  location → Why Join Us). The skills query scores **0.71 on Qualifications vs 0.46 on the whole-JD vector** —
  that gap is chunking working. The two vague queries win by small margins only (vague query → vague match).
- Arbeitnow: the median job keeps only ~3,000 visible chars of its 4,000 → ~25% of the window is HTML.
  ~80% of Arbeitnow jobs are cut off.
- Cheap fixes (Chat 28/30, measured first): strip HTML before embedding; embed sections (or the requirements part
  first); report the truncation rate as a metric.

Two words for later: a search over chunks returns chunks, so you **roll up** to jobs (max chunk score per job = "max-score
roll-up" in the roadmap). And chunk size is a trade-off — too small loses context, too big blurs topics.

---
## Step 4 — Mini-RAG: "why does this job match me?"

**RAG** (retrieval-augmented generation) = *retrieve* the relevant text first, *put it in the prompt*, then let the
model answer **from that text**. The LLM is the writer, retrieval is the research. Without retrieval the model can only
guess from the title — which is how hallucinations happen.

Same question, same prompt, three different contexts:
- **A** — title + company only (no retrieval)
- **B** — what the pipeline has today: the first 4,000 raw chars (HTML stripped)
- **C** — retrieved sections from Step 3: Responsibilities + Qualifications + Nice-to-have

In [7]:
prof = yaml.safe_load(open(REPO / "config" / "user_profile.yml"))["profile"]
me = (f"Skills: {', '.join(prof['skills'])}. Experience: {prof['yoe']} years. "
      f"Preferred locations: {', '.join(prof['preferred_locations'])}.")

SYSTEM = ("You explain job matches to a candidate. Answer in at most 6 short bullets. "
          "Use only the job text you are given. If it does not say something, write 'not stated' instead of guessing.")

def ask(job_text):
    r = claude.messages.create(
        model="claude-haiku-5-5", max_tokens=2000, output_config={"effort": "low"}, system=SYSTEM,
        messages=[{"role": "user", "content": f"My profile: {me}\n\n<job>\n{job_text}\n</job>\n\n"
                                              "Which skills does this job ask for that I have, which am I missing, "
                                              "and is the seniority right for me?"}])
    answer = "".join(b.text for b in r.content if b.type == "text")
    cost = r.usage.input_tokens * 0.10 / 1e6 + r.usage.output_tokens * 0.50 / 1e6
    return answer, r.usage.input_tokens, r.usage.output_tokens, cost

contexts = {
    "A — title only": f"{jd['title']} at {jd['company_name']}",
    "B — pipeline's first 4,000 chars": strip_html(raw[:MAX_CHARS]),
    "C — retrieved sections": "\n\n".join(sec.set_index("section").loc[["Responsibilities", "Qualifications", "Nice-to-have"], "text"]),
}
total = 0.0
for name, ctx in contexts.items():
    answer, tin, tout, cost = ask(ctx)
    total += cost
    display(Markdown(f"### {name}\n*context {len(ctx):,} chars · {tin:,} in / {tout:,} out tokens · ${cost:.5f}*\n\n{answer}"))
print(f"total Claude cost for this step: ${total:.4f}")

### A — title only
*context 30 chars · 275 in / 434 out tokens · $0.00024*

- **Skills the job asks for:** not stated. The job text only gives the title "Senior Data Engineer" and the company "Aqemia."
- **Skills you're missing:** not stated. Without a requirements list, I can't compare your skills to the role.
- **Seniority fit:** not stated. The posting doesn't list required years of experience or responsibilities. The "Senior" title suggests a more experienced level than your 2 years, but that comes from the title alone.
- **Location:** not stated. The posting doesn't mention where the role is based or whether it's remote, so I can't check it against your preferred locations.
- **Next step:** paste the full job description, including requirements, responsibilities, and location, and I'll compare it with your profile.

### B — pipeline's first 4,000 chars
*context 3,323 chars · 1,288 in / 806 out tokens · $0.00053*

- **Overlapping skills:** The posting doesn't list a skills section. Its responsibilities point to data modeling, owning a data platform end to end, and building pipelines from ingestion to model-ready datasets. Your profile matches these through data modeling, plus pipeline tools like airflow, spark, pyspark, and kafka, and warehouses like snowflake, bigquery, and databricks.
- **Likely gaps:** The posting says the role centers on canonical identity, provenance, and lineage across messy scientific data. Your profile doesn't mention lineage or provenance tooling, such as OpenLineage or data catalogs. It also doesn't mention chemistry or cheminformatics (e.g., RDKit or molecular data formats).
- **Not stated:** Required tools, cloud platform, and any specific technical requirements. The text cuts off at "You'll work at," so the rest of the role details aren't visible.
- **Seniority fit:** The title is "Senior Data Engineer," but the required years of experience are not stated. Your profile shows 2 years, so you'd need to judge from the full posting whether that meets their bar.
- **Location:** The posting names Paris and London as where the team is based, but the role's location and remote policy are not stated. Your preferences are remote, India, Bangalore, and Mumbai, so confirm this before applying.
- **Next step:** Get the full posting, since the visible text is incomplete. Then check the requirements and location details against this summary.

### C — retrieved sections
*context 2,475 chars · 1,046 in / 692 out tokens · $0.00045*

- **Matches:** Python, SQL, data modeling, Snowflake, dbt, Airflow, Terraform, and AWS all appear in both your profile and the job's core or nice-to-have lists. Your BigQuery and Redshift experience also matches the nice-to-have warehouse examples.
- **Missing or not shown in your profile:** Drug discovery, biotech, pharma, or deeptech experience. Data governance, lineage, and metadata management tooling. Object storage experience is not stated explicitly, though Delta Lake and Iceberg may partly cover it.
- **Seniority:** The role asks for "deep experience" with production systems other teams depend on, plus mentoring and acting as a senior individual contributor. Your profile shows 2 years of experience. The posting says it cares more about what you've built than the year count, so your project track record will matter more than the number.
- **Strongest signal to show:** Production pipelines you own end to end, especially with testing, data quality monitoring, and identity or lineage decisions.
- **Location:** The job text does not state a location or remote policy, so I cannot confirm it fits your preferred locations (remote, India, Bangalore, Mumbai).
- **Compensation and team size:** Not stated in the job text.

total Claude cost for this step: $0.0012


**What to notice**
- **A** has nothing to go on — with the "say not stated" rule it mostly refuses; without that rule it would invent a
  plausible DE skill list. Plausible ≠ true.
- **B** is the trap: lots of text, little signal. The model sees a company story and almost no skills — exactly what
  the pipeline's vector sees.
- **C** names this job's real requirements. Fewer tokens than B, better answer: **retrieval quality decides answer
  quality.** "Grounded" = every claim traceable to the given text. Chat 32's weekly brief must work like C.

---
## Step 5 — Precision@10

**precision@10** = of the top 10 results, how many are relevant? 7 good ones → 0.7. It answers "is page 1 useful?".
It ignores *order* inside the 10 (nDCG does that, Chat 28) and ignores the good jobs you never showed (recall).

**Labels** are the human judgement "is this job relevant to this query" (0/1 here; 0/1/2 in Chat 27).
Below are **Claude's draft labels, from titles only**. Your exercise: read the snippets, change any label you disagree
with, re-run, and see how much one label moves the score.

In [8]:
labels = {  # 1 = relevant, 0 = not.  Unlisted job = 0.
    "data engineer building pipelines with spark, airflow and python": {
        "data-engineerin-hamburg-129224": 1, "5903289517": 1, "1171507875": 1, "data-engineer-experimentee-lille-351246": 1,
        "5917977915": 1, "senior-data-engineer-f-h-rennes-49167": 1, "senior-data-engineer-f-h-niort-287138": 1,
        "senior-data-engineer-f-h-nantes-365382": 1, "5887898278": 0, "data-engineer-b-toulouse-333961": 1, "5906753776": 1,
        "5879360387": 0, "backend-engineer-commodities-united-kingdom-202132": 0, "2091132": 0,
        "senior-full-stack-engineer-hawkai-budapest-73855": 0, "5887141857": 0,
        "template-of-member-of-technical-staff-full-stack-paris-423144": 0, "data-scientist-london-430386": 0,
    },
    "devops engineer with kubernetes and terraform": {
        "2294895246": 1, "5870612725": 1, "senior-devops-engineer-f-h-creation-dagence-niort-325065": 1,
        "platform-engineer-kubernetes-automation-wurzburg-82406": 1, "infrastructure-engineer-platform-paris-93129": 1,
        "5871278855": 1, "5906316723": 1, "5906316712": 1, "remote-junior-cloud-operations-engineer-all-genders-ch-baar-284023": 1,
        "ingenieure-devops-cloud-experimentee-lille-170227": 1, "2091132": 0, "5899028943": 0, "5903336647": 0,
        "5797270399": 0, "customer-success-engineer-germany-80446": 0,
    },
}

def top10(query):
    q = embed([query])[0]
    return {"vector": jobs["job_id"].iloc[np.argsort(-(M @ q))[:10]].tolist(),
            "BM25": jobs["job_id"].iloc[np.argsort(-bm25.get_scores(tokens(query)))[:10]].tolist()}

by_id = jobs.set_index("job_id")
scores = []
for query, lab in labels.items():
    results = top10(query)
    for method, ids in results.items():
        missing = [j for j in ids if j not in lab]
        scores.append({"query": query, "method": method, "precision@10": sum(lab.get(j, 0) for j in ids) / 10,
                       "unlabelled_in_top10": len(missing)})
    pool = list(dict.fromkeys(results["vector"] + results["BM25"]))  # union, keeps order
    display(Markdown(f"**Pool to label for:** `{query}`"))
    display(pd.DataFrame({"label": [lab.get(j, "?") for j in pool],
                          "in": ["V+B" if j in results["vector"] and j in results["BM25"] else "V" if j in results["vector"] else "B" for j in pool],
                          "title": by_id.loc[pool, "title"].values,
                          "snippet": [strip_html(d)[:140] for d in by_id.loc[pool, "description"]]}, index=pool))

pd.DataFrame(scores).pivot(index="query", columns="method", values="precision@10")

**Pool to label for:** `data engineer building pipelines with spark, airflow and python`

,label,in,title,snippet
data-engineerin-hamburg-129224,1,V,DATA ENGINEER*IN (m/w/d),Dein Team Das Data & Automation Team ist Teil der OMR Tech Abteilung und fun...
5903289517,1,V+B,Data Engineer (Full Stack) | Contract,We are seeking an experienced Data Engineer (Full Stack) to join a dynamic d...
1171507875,1,V+B,Bigdata Lead,Description Deep experience and understanding of Apache Hadoop and surroundi...
data-engineer-experimentee-lille-351246,1,V,Data Engineer expérimenté(e),LILLE | AI-Native Group 🤖 Sharp Tech. Real Impact. 🧠 Ce qu'on recherche chez...
5917977915,1,V+B,Senior Data Engineer,Senior Data Engineer 12 – month max term – Brisbane QLD Contract | Max-term ...
senior-data-engineer-f-h-rennes-49167,1,V,Senior Data Engineer (F/H) - Rennes,🚀 Senior Data Engineer (F/H) – Création d'agence Rennes 👾 Tu as l'âme d'un v...
senior-data-engineer-f-h-niort-287138,1,V,Senior Data Engineer (F/H) - Niort,🚀 Senior Data Engineer (F/H) 👾 Tu as l'âme d'un vibe coder ? Tu développes a...
senior-data-engineer-f-h-nantes-365382,1,V,Senior Data Engineer (F/H) - Nantes,👾 Tu as l’âme d’un vibe coder ? Tu développes avec Claude Code ouvert en per...
5887898278,0,V,AI / ML Engineer,Project Role : AI / ML Engineer Project Role Description : Develops applicat...
5906753776,1,V,Data Engineer,"Our client, a well-established South African organisation operating in the f..."


**Pool to label for:** `devops engineer with kubernetes and terraform`

,label,in,title,snippet
2294895246,1,V+B,Devops Engineer,[Devops Engineer] Experience - 3 JD * Experience working on Linux based infr...
5870612725,1,V+B,Azure IAC (Terraform) Engineer,"Summary Avanade is growing, and we seek motivated and engaged technology pro..."
platform-engineer-kubernetes-automation-wurzburg-82406,1,V,Platform Engineer Kubernetes & Automation (m/w/d),Die AXSOL GmbH entwickelt innovative Batteriespeichersysteme für unterschied...
senior-devops-engineer-f-h-creation-dagence-niort-325065,1,V+B,Senior DevOps Engineer (F/H) – Création d’agence Niort,"🚀 Senior DevOps Engineer – Création d’agence SFEIR Niort ☁️ Pour toi, le Dev..."
infrastructure-engineer-platform-paris-93129,1,V,Infrastructure Engineer (Platform) - Paris,You'll be joining the team responsible for the infrastructure behind all of ...
5871278855,1,V,Cloud Engineer,"Key Responsibilities Administer and maintain cloud infrastructure, including..."
5906316723,1,V+B,Senior DevOps Engineer – 10 Yrs - Johannesburg (Hybrid),Senior DevOps Engineer – 10 Yrs - Johannesburg (Hybrid) Are you a talented D...
5906316712,1,V+B,Senior DevOps Engineer – 10 Yrs - Johannesburg (In office),Senior DevOps Engineer – 10 Yrs - Johannesburg (In office) Are you a talente...
remote-junior-cloud-operations-engineer-all-genders-ch-baar-284023,1,V,Junior Cloud Operations Engineer (all genders) CH,Alle reden von Digitalisierung – Wir machen es. Als Hersteller von Europas #...
ingenieure-devops-cloud-experimentee-lille-170227,1,V,Ingénieur(e) DevOps/cloud expérimenté(e),"👾 Tu as l’âme d’un vibe coder, tu prompts plus vite que ton ombre et tu pens..."


method,BM25,vector
query,,
"data engineer building pipelines with spark, airflow and python",0.3,0.9
devops engineer with kubernetes and terraform,0.5,1.0


**What to notice**
- Two queries and title-only labels are a *demo*, not a result. One relabel changes a score by 0.1 — that's why
  Chat 27 builds ~50 queries and ~300 hand labels and checks a Claude judge against you (Cohen's κ).
- **Pooling** = labelling only the union of what the systems returned (the table above), not all 1,544 jobs. Chat 27 does the same.
- Vector beats BM25 on these *descriptive* queries. Step 2 showed the reverse for *exact tool names*. A good query set
  needs both kinds — or the eval just confirms whatever you hoped.

---
## Say it in 90 seconds (practice out loud)
1. **Embedding** — "text → 1,024 numbers; similar meaning → small angle. Our nightly Glue job embeds each JD with Voyage
   and stores the vectors as Parquet next to gold."
2. **Keyword vs vector** — "BM25 finds exact words, vectors find meaning. 'Snowflake' gives Christmas staff in vector
   search, 'keeps servers running' gives housekeeping in BM25 — so we go hybrid and merge by rank."
3. **Chunking** — "We embed the first 4,000 raw chars. For a real Senior DE job that's the company blurb plus HTML;
   the requirements start at char 4,371. Section chunks put each query on the right section."
4. **RAG** — "Same question, three contexts: title only guesses, the truncated blurb can't see skills, the retrieved
   sections answer correctly with fewer tokens. Retrieval quality decides answer quality."
5. **Precision@10** — "Of the top 10, how many are relevant. Needs labelled queries; two queries is a demo — the
   real eval is Chat 27."